In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

tf.keras.backend.clear_session()

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("TensorFlow Version:", tf.__version__)


TensorFlow Version: 2.19.0


In [ ]:
(ds_train, ds_val), ds_info = tfds.load(
    "imagenette/160px",
    split=["train", "validation"],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features["label"].num_classes
print("Number of classes:", NUM_CLASSES)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imagenette/160px/incomplete.HBRPOI_1.0.0/imagenette-train.tfrecord*...:   …

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imagenette/160px/incomplete.HBRPOI_1.0.0/imagenette-validation.tfrecord*..…

Dataset imagenette downloaded and prepared to /root/tensorflow_datasets/imagenette/160px/1.0.0. Subsequent calls will reuse this data.
Number of classes: 10


In [ ]:
IMG_SIZE = (227, 227)
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

train_ds = ds_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds   = ds_val.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [ ]:
def build_alexnet(input_shape=(227, 227, 3), num_classes=10):
    model = Sequential(name="AlexNet")

    model.add(Conv2D(96, (11,11), strides=4, activation="relu",
                     input_shape=input_shape))
    model.add(MaxPooling2D((3,3), strides=2))

    model.add(Conv2D(256, (5,5), padding="same", activation="relu"))
    model.add(MaxPooling2D((3,3), strides=2))

    model.add(Conv2D(384, (3,3), padding="same", activation="relu"))
    model.add(Conv2D(384, (3,3), padding="same", activation="relu"))
    model.add(Conv2D(256, (3,3), padding="same", activation="relu"))
    model.add(MaxPooling2D((3,3), strides=2))

    model.add(Flatten())
    model.add(Dense(4096, activation="relu"))
    model.add(Dropout(0.5))
    model.add(Dense(4096, activation="relu"))
    model.add(Dropout(0.5))
    model.add(Dense(num_classes, activation="softmax"))

    return model


In [ ]:
alexnet = build_alexnet()

alexnet.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

alexnet.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "AlexNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 55, 55, 96)     │        34,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 27, 27, 96)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 27, 27, 256)    │       614,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 13, 13, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 13, 13, 384)    │       885,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 13, 13, 384)    │     1,327,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 13, 13, 256)    │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4096)           │    37,752,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │        40,970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,322,314 (222.48 MB)

 Trainable params: 58,322,314 (222.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
EPOCHS = 5

history = alexnet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)


Epoch 1/5
403/403 ━━━━━━━━━━━━━━━━━━━━ 45s 80ms/step - accuracy: 0.2100 - loss: 2.1259 - val_accuracy: 0.4520 - val_loss: 1.5870
Epoch 2/5
403/403 ━━━━━━━━━━━━━━━━━━━━ 23s 55ms/step - accuracy: 0.5020 - loss: 1.4910 - val_accuracy: 0.6560 - val_loss: 1.0623
Epoch 3/5
403/403 ━━━━━━━━━━━━━━━━━━━━ 41s 55ms/step - accuracy: 0.6362 - loss: 1.1006 - val_accuracy: 0.7100 - val_loss: 0.8689
Epoch 4/5
403/403 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.7113 - loss: 0.8869 - val_accuracy: 0.7480 - val_loss: 0.7830
Epoch 5/5
403/403 ━━━━━━━━━━━━━━━━━━━━ 23s 56ms/step - accuracy: 0.7531 - loss: 0.7500 - val_accuracy: 0.7660 - val_loss: 0.7169


In [ ]:
loss, acc = alexnet.evaluate(val_ds)

print("\n===== RESULTS =====")
print(f"Validation Accuracy : {acc * 100:.2f}%")
print(f"Validation Loss     : {loss:.4f}")


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 0.7714 - loss: 0.7479

===== RESULTS =====
Validation Accuracy : 76.60%
Validation Loss     : 0.7169


In [ ]:
print("\n===== MODEL DETAILS =====")
print("Total Parameters :", f"{alexnet.count_params():,}")
print("Total Layers     :", len(alexnet.layers))



===== MODEL DETAILS =====
Total Parameters : 58,322,314
Total Layers     : 14
